# Valley Bottom Extraction (VBET)
## Maximum Riparian Corridor Extent — Cheyenne River, SD

This notebook delineates the valley bottom (maximum riparian corridor extent) for a user-defined
reach of the Cheyenne River, South Dakota. The output polygon serves as the study area mask
for cottonwood and invasive riparian species classification in later notebooks.

### Methodology
Adapted from the [VBET-2](https://github.com/jtgilbert/VBET-2) open-source tool and
Woodward et al. (2018) CO-RIP methodology. Core metric is
**HAND (Height Above Nearest Drainage)** — the vertical distance from each terrain cell
to its nearest stream cell along the flow path. Valley bottoms have low HAND values;
hillslopes have high values.

### Pipeline
1. AOI definition + NHD flowlines
2. DEM acquisition (USGS 3DEP via `py3dep`)
3. Hydrological conditioning (`pysheds`)
4. HAND computation
5. Valley bottom delineation (drainage-area-aware thresholds)
6. Post-processing + export
7. Visualization & validation

### References
- Woodward, B.D. et al. (2018). CO-RIP: A Riparian Vegetation and Corridor Extent Dataset. *ISPRS Int. J. Geo-Inf.*, 7(10), 397. https://doi.org/10.3390/ijgi7100397
- Gilbert, J.T. VBET-2. https://github.com/jtgilbert/VBET-2
- Nobre et al. (2011). Height Above the Nearest Drainage. *J. Hydrology*, 404(1–2), 13–29.

## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path

import rasterio
from rasterio.features import shapes
from rasterio.transform import from_bounds
import rioxarray as rxr
import xarray as xr

from shapely.geometry import box, shape, mapping
from shapely.ops import unary_union

import folium
from pynhd import NLDI

# These must be installed: py3dep, pysheds, scipy
import py3dep
from pysheds.grid import Grid
from scipy.ndimage import label, binary_fill_holes

# Output directory
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

print("Imports OK")

---
## 1. AOI Configuration

**Edit this cell to define your study reach.**

Three options (uncomment the one you want):
- **Option A**: Bounding box in WGS84 (fastest)
- **Option B**: Path to an existing polygon file (e.g., a prior study area)
- **Option C**: USGS gauge ID + upstream/downstream distance — builds AOI from the NHD basin

The `BUFFER_KM` parameter adds a buffer around flowlines before downloading the DEM
to ensure full valley width is captured. 5–8 km is appropriate for the Cheyenne River main stem.

In [ ]:
# ============================================================
# CONFIGURE YOUR AOI HERE
# ============================================================

# --- Option A: Bounding box (xmin, ymin, xmax, ymax) in WGS84 ---
# Covers ~100 km of the Cheyenne River main stem around the Red Shirt gauge
# AOI_BBOX = (-103.2, 43.5, -101.8, 44.2)

# --- Option B: Existing polygon file ---
# AOI_FILE = DATA_DIR / "CravenCanyon_StudyArea.gpkg"

# --- Option C: USGS gauge + navigate upstream/downstream (km) ---
GAUGE_ID = "06403700"        # Cheyenne River at Red Shirt, SD
UPSTREAM_KM = 150            # km upstream from gauge to include
DOWNSTREAM_KM = 50           # km downstream from gauge to include

# Buffer added around flowlines for DEM download (km)
BUFFER_KM = 6

# DEM resolution: 10 (m) for local/regional, 30 (m) for very large AOIs
DEM_RESOLUTION = 10

# Flow accumulation threshold for defining streams (number of 10m cells)
# 500 cells @ 10m = ~50 km² contributing area
FLOW_ACCUM_THRESHOLD = 500

# VBET thresholds by drainage area class
# Each entry: (min_km2, max_km2, hand_threshold_m, slope_threshold_deg, buffer_m)
VBET_CLASSES = [
    {"label": "large",  "da_min": 1000,  "da_max": 1e9,  "hand_m": 12, "slope_deg": 6,  "buffer_m": 1000},
    {"label": "medium", "da_min": 100,   "da_max": 1000, "hand_m": 8,  "slope_deg": 8,  "buffer_m": 500},
    {"label": "small",  "da_min": 0,     "da_max": 100,  "hand_m": 4,  "slope_deg": 12, "buffer_m": 200},
]

# Minimum valley bottom patch size to keep (hectares)
MIN_PATCH_HA = 1.0

# Output file
OUTPUT_GPKG = DATA_DIR / "cheyenne_valley_bottom.gpkg"

print("Configuration set")

---
## 1. AOI Definition & Flowlines

Build the AOI polygon and retrieve NHD flowlines. The flowlines are used to:
1. Define the DEM download extent (buffered bounding box)
2. Assign drainage area classes to stream reaches (for VBET thresholding)

In [ ]:
nldi = NLDI()

# ---- Build AOI from selected option ----
if 'AOI_BBOX' in dir() or 'AOI_BBOX' in globals():
    aoi_geom = box(*AOI_BBOX)
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom], crs="EPSG:4326")
    print(f"Using bounding box AOI: {AOI_BBOX}")

elif 'AOI_FILE' in dir() or 'AOI_FILE' in globals():
    aoi_gdf = gpd.read_file(AOI_FILE).to_crs(4326)
    aoi_geom = aoi_gdf.union_all()
    print(f"Using polygon AOI from: {AOI_FILE}")

else:
    # Option C: gauge-based AOI
    print(f"Fetching NHD data for gauge {GAUGE_ID}...")
    station = nldi.getfeature_byid("nwissite", f"USGS-{GAUGE_ID}").to_crs(4326)

    flw_us = nldi.navigate_byid(
        fsource="nwissite", fid=f"USGS-{GAUGE_ID}",
        navigation="upstreamMain", source="flowlines", distance=UPSTREAM_KM,
    )
    flw_ds = nldi.navigate_byid(
        fsource="nwissite", fid=f"USGS-{GAUGE_ID}",
        navigation="downstreamMain", source="flowlines", distance=DOWNSTREAM_KM,
    )
    flw_trib = nldi.navigate_byid(
        fsource="nwissite", fid=f"USGS-{GAUGE_ID}",
        navigation="upstreamTributaries", source="flowlines", distance=UPSTREAM_KM,
    )

    # Combine all flowlines
    flowlines = pd.concat([flw_us, flw_ds, flw_trib], ignore_index=True)
    flowlines = gpd.GeoDataFrame(flowlines, crs="EPSG:4326").drop_duplicates(subset="geometry")

    # AOI = bounding box of flowlines
    aoi_geom = box(*flowlines.total_bounds)
    aoi_gdf = gpd.GeoDataFrame(geometry=[aoi_geom], crs="EPSG:4326")
    print(f"  Flowlines retrieved: {len(flowlines)} segments")

# Project to UTM for metric operations (UTM zone 13N covers the Cheyenne River)
CRS_PROJ = "EPSG:32613"   # UTM Zone 13N
aoi_proj = aoi_gdf.to_crs(CRS_PROJ)

# Buffer the AOI for DEM download to capture full valley walls
buffer_m = BUFFER_KM * 1000
dem_aoi_proj = aoi_proj.copy()
dem_aoi_proj.geometry = aoi_proj.geometry.buffer(buffer_m)
dem_aoi_wgs84 = dem_aoi_proj.to_crs(4326)
dem_aoi_geom = dem_aoi_wgs84.union_all()

print(f"AOI extent (WGS84): {dem_aoi_geom.bounds}")
print(f"AOI area: {dem_aoi_proj.area.sum() / 1e6:.0f} km²")

In [ ]:
# If flowlines weren't fetched above (Option A/B), fetch them now clipped to AOI
if 'flowlines' not in dir() and 'flowlines' not in globals():
    print(f"Fetching NHD flowlines within AOI...")
    station = nldi.getfeature_byid("nwissite", f"USGS-{GAUGE_ID}").to_crs(4326)
    flw_us = nldi.navigate_byid(
        fsource="nwissite", fid=f"USGS-{GAUGE_ID}",
        navigation="upstreamMain", source="flowlines", distance=UPSTREAM_KM,
    )
    flw_trib = nldi.navigate_byid(
        fsource="nwissite", fid=f"USGS-{GAUGE_ID}",
        navigation="upstreamTributaries", source="flowlines", distance=UPSTREAM_KM,
    )
    flowlines = pd.concat([flw_us, flw_trib], ignore_index=True)
    flowlines = gpd.GeoDataFrame(flowlines, crs="EPSG:4326").drop_duplicates(subset="geometry")

# Clip flowlines to AOI
flowlines_clipped = flowlines.clip(dem_aoi_wgs84)
flowlines_proj = flowlines_clipped.to_crs(CRS_PROJ)
print(f"Flowlines in AOI: {len(flowlines_proj)} segments")

---
## 2. DEM Acquisition via py3dep

Fetch USGS 3DEP DEM for the buffered AOI. For large areas (> ~2000 km²), the 10m DEM will
be several GB — `py3dep` handles this via server-side mosaicking.

If the download fails or times out for a very large AOI, reduce `DEM_RESOLUTION` to 30.

In [ ]:
dem_path = DATA_DIR / f"cheyenne_dem_{DEM_RESOLUTION}m.tif"

if dem_path.exists():
    print(f"DEM already exists at {dem_path}, skipping download")
    dem_da = rxr.open_rasterio(dem_path, masked=True).squeeze()
else:
    print(f"Downloading {DEM_RESOLUTION}m DEM from USGS 3DEP...")
    print("  This may take several minutes for large AOIs.")

    dem_da = py3dep.get_dem(
        geometry=dem_aoi_geom,
        resolution=DEM_RESOLUTION,
        crs="EPSG:4326",
    )

    # Reproject to projected CRS for metric analysis
    dem_da = dem_da.rio.reproject(CRS_PROJ)

    # Save to disk so we don't re-download
    dem_da.rio.to_raster(dem_path, compress="deflate", dtype="float32")
    print(f"  Saved to {dem_path}")

print(f"DEM shape: {dem_da.shape}")
print(f"DEM resolution: {abs(float(dem_da.rio.resolution()[0])):.1f} m")
print(f"Elevation range: {float(dem_da.min()):.1f} – {float(dem_da.max()):.1f} m")

In [ ]:
# Quick hillshade preview
fig, ax = plt.subplots(figsize=(12, 6))
dem_da.plot(ax=ax, cmap="terrain", robust=True, cbar_kwargs={"label": "Elevation (m)"})
flowlines_proj.plot(ax=ax, color="blue", linewidth=0.5, label="Flowlines")
ax.set_title("DEM with NHD Flowlines")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

---
## 3. Hydrological Conditioning (pysheds)

Condition the DEM to ensure hydrological connectivity:
1. **Fill pits** — remove single-cell depressions (data artifacts)
2. **Fill depressions** — fill enclosed basins that would trap water
3. **Resolve flats** — assign flow direction across flat areas
4. **Flow direction** — D8 algorithm (each cell drains to lowest neighbor)
5. **Flow accumulation** — count of upstream contributing cells

In [ ]:
dem_array_path = str(dem_path)
print("Loading DEM into pysheds Grid...")
grid = Grid.from_raster(dem_array_path)
dem = grid.read_raster(dem_array_path)

print("Filling pits...")
pit_filled = grid.fill_pits(dem)

print("Filling depressions...")
flooded = grid.fill_depressions(pit_filled)

print("Resolving flats...")
inflated = grid.resolve_flats(flooded)

print("Computing flow direction (D8)...")
fdir = grid.flowdir(inflated)

print("Computing flow accumulation...")
acc = grid.accumulation(fdir)

print(f"Max flow accumulation: {int(acc.max())} cells")
print("Hydrological conditioning complete")

In [ ]:
# Preview flow accumulation (log scale reveals stream network)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].imshow(np.log1p(acc), cmap="Blues", origin="upper")
axes[0].set_title(f"Flow Accumulation (log scale)\nStream threshold ≥ {FLOW_ACCUM_THRESHOLD} cells")

stream_mask = acc > FLOW_ACCUM_THRESHOLD
axes[1].imshow(stream_mask, cmap="binary", origin="upper")
axes[1].set_title("Derived Stream Network")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Stream cells: {stream_mask.sum():,} ({stream_mask.sum() * DEM_RESOLUTION**2 / 1e6:.0f} km² of streams)")

---
## 4. HAND Computation (Height Above Nearest Drainage)

HAND measures the vertical distance from each terrain cell to the elevation of its nearest
stream cell, traced along the flow path. Cells in the valley bottom (floodplain) have low
HAND values; hillslopes and uplands have high values.

Formula: `HAND[cell] = elevation[cell] − elevation[nearest_stream_cell_along_flow_path]`

Reference: Nobre et al. (2011), *J. Hydrology*, 404(1–2).

In [ ]:
hand_path = DATA_DIR / f"cheyenne_hand_{DEM_RESOLUTION}m.tif"

if hand_path.exists():
    print(f"HAND already exists at {hand_path}, loading...")
    hand_da = rxr.open_rasterio(hand_path, masked=True).squeeze()
    hand = np.array(hand_da.values)
else:
    print("Computing HAND (Height Above Nearest Drainage)...")
    print("  This is the most compute-intensive step — may take several minutes.")

    # pysheds >= 0.4 has compute_hand(); for older versions we use
    # hand = grid.compute_hand(fdir=fdir, dem=inflated, mask=stream_mask)
    try:
        hand = grid.compute_hand(fdir=fdir, dem=inflated, mask=stream_mask)
        print("  Used grid.compute_hand()")
    except AttributeError:
        # Fallback for older pysheds: use snap_to_mask approach
        print("  grid.compute_hand not found, using distance_to_outlet approach...")
        # Get elevation at the nearest drainage cell for each cell
        # by snapping non-stream cells to the stream network
        nearest_stream_elev = grid.snap_to_mask(stream_mask, inflated, fdir)
        hand = np.where(stream_mask, 0.0, np.array(inflated) - np.array(nearest_stream_elev))
        hand = np.clip(hand, 0, None)   # HAND is always >= 0

    # Save HAND raster (reuse DEM geotransform/CRS)
    hand_array = np.array(hand, dtype=np.float32)
    with rasterio.open(dem_path) as src:
        profile = src.profile.copy()
        profile.update(dtype="float32", count=1, compress="deflate", nodata=-9999.0)
        with rasterio.open(hand_path, "w", **profile) as dst:
            dst.write(hand_array[np.newaxis, :, :])
    print(f"  HAND saved to {hand_path}")

    hand_da = rxr.open_rasterio(hand_path, masked=True).squeeze()

print(f"HAND range: 0 – {float(np.nanpercentile(hand_da, 99)):.1f} m (99th pct)")

In [ ]:
# HAND visualization — low values = valley bottom, high = hillslope
fig, ax = plt.subplots(figsize=(14, 6))
vmax = float(np.nanpercentile(hand_da, 95))
hand_da.plot(ax=ax, cmap="RdYlGn_r", vmin=0, vmax=vmax,
             cbar_kwargs={"label": "HAND (m above nearest drainage)"})
flowlines_proj.plot(ax=ax, color="blue", linewidth=0.5, label="NHD Flowlines")
ax.set_title("Height Above Nearest Drainage (HAND)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

---
## 5. Valley Bottom Delineation — Simplified VBET-2 Logic

Apply drainage-area-aware HAND and slope thresholds following VBET-2.

**Rationale**: Large rivers occupy broad, flat floodplains (higher HAND threshold
allowable); small headwater streams are confined to narrow steep valleys (lower threshold).
Each NHD stream reach is assigned a drainage area class and buffered accordingly, then only
pixels within the buffer that satisfy both the HAND and slope criteria are retained.

**Thresholds** (adjustable in the configuration cell above):

| Class | Drainage Area | Max HAND | Max Slope | Search Buffer |
|-------|--------------|----------|-----------|---------------|
| Large | > 1000 km² | 12 m | 6° | 1000 m |
| Medium | 100–1000 km² | 8 m | 8° | 500 m |
| Small | < 100 km² | 4 m | 12° | 200 m |

In [ ]:
# ---- Compute slope from DEM (degrees) ----
print("Computing slope...")
with rasterio.open(dem_path) as src:
    dem_arr = src.read(1).astype(np.float32)
    res = abs(src.transform[0])  # cell size in metres

# Finite-difference gradient → slope in degrees
dz_dy, dz_dx = np.gradient(dem_arr, res, res)
slope_arr = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

# Load HAND as numpy array aligned with DEM
with rasterio.open(hand_path) as src_h:
    hand_arr = src_h.read(1).astype(np.float32)
    hand_arr = np.where(hand_arr < 0, np.nan, hand_arr)  # mask nodata

print(f"  Slope range: 0 – {np.nanpercentile(slope_arr, 99):.1f}°")

In [ ]:
from rasterio.transform import rowcol
from rasterio.features import rasterize

# ---- Assign drainage area classes to flowlines ----
# NHDPlus flowlines include 'totdasqkm' (total drainage area in km²)
# Fall back to estimating from flow accumulation if not present

with rasterio.open(dem_path) as src:
    raster_meta = src.meta.copy()
    raster_transform = src.transform
    raster_shape = (src.height, src.width)

# Build a combined valley bottom mask (starts empty)
valley_mask = np.zeros(raster_shape, dtype=np.uint8)

for cls in VBET_CLASSES:
    da_col = None
    for col in ["totdasqkm", "totDASqKm", "drainage_area_km2"]:
        if col in flowlines_proj.columns:
            da_col = col
            break

    if da_col:
        class_flw = flowlines_proj[
            (flowlines_proj[da_col] >= cls["da_min"]) &
            (flowlines_proj[da_col] < cls["da_max"])
        ]
    else:
        # No drainage area attribute — estimate from flow accumulation
        # Use cell_area * threshold as proxy; map back via spatial join
        # For now, assign everything to 'medium' if no da attribute found
        if cls["label"] == "medium":
            class_flw = flowlines_proj
        else:
            continue
        print(f"  Warning: no drainage area column found; all reaches assigned to 'medium'")

    if len(class_flw) == 0:
        print(f"  Class '{cls['label']}': no reaches, skipping")
        continue

    # Buffer flowlines by class-specific distance
    buffered = class_flw.geometry.buffer(cls["buffer_m"])
    buffered_union = buffered.union_all()

    # Rasterize the buffer polygon
    buffer_raster = rasterize(
        [(mapping(buffered_union), 1)],
        out_shape=raster_shape,
        transform=raster_transform,
        fill=0,
        dtype=np.uint8,
    )

    # Valley bottom = in buffer AND HAND < threshold AND slope < threshold
    class_mask = (
        (buffer_raster == 1) &
        (hand_arr < cls["hand_m"]) &
        (slope_arr < cls["slope_deg"]) &
        np.isfinite(hand_arr)
    ).astype(np.uint8)

    valley_mask = np.maximum(valley_mask, class_mask)
    print(f"  Class '{cls['label']}': {int(class_mask.sum()):,} cells, "
          f"{int(class_mask.sum()) * res**2 / 1e4:.0f} ha")

print(f"Combined valley mask: {int(valley_mask.sum()):,} cells "
      f"({int(valley_mask.sum()) * res**2 / 1e6:.1f} km²)")

---
## 6. Post-Processing & Export

1. **Fill holes** — fill enclosed upland patches inside the valley bottom
2. **Remove slivers** — drop isolated patches smaller than `MIN_PATCH_HA`
3. **Smooth boundaries** — buffer then un-buffer to remove jagged edges
4. **Vectorize** — convert raster mask to polygon
5. **Export** — save as GeoPackage

In [ ]:
# ---- Morphological cleanup ----
print("Post-processing valley mask...")

# Fill small holes in the valley bottom
valley_filled = binary_fill_holes(valley_mask).astype(np.uint8)

# Label connected components and remove patches below minimum area
labeled, n_features = label(valley_filled)
print(f"  Connected components before filtering: {n_features}")

min_cells = int((MIN_PATCH_HA * 1e4) / (res**2))  # ha → cell count
for i in range(1, n_features + 1):
    if (labeled == i).sum() < min_cells:
        valley_filled[labeled == i] = 0

labeled_clean, n_clean = label(valley_filled)
print(f"  Components after size filter (≥ {MIN_PATCH_HA} ha): {n_clean}")

# ---- Vectorize to polygon ----
print("Vectorizing...")
with rasterio.open(dem_path) as src:
    transform = src.transform
    crs = src.crs

polygons = [
    shape(geom)
    for geom, val in shapes(valley_filled, transform=transform)
    if val == 1
]
valley_poly = unary_union(polygons)

# Smooth boundaries (buffer + un-buffer removes pixel-edge jaggedness)
smooth_dist = res * 2   # 2-cell smooth
valley_smooth = valley_poly.buffer(smooth_dist).buffer(-smooth_dist)

print(f"  Final valley bottom area: {valley_smooth.area / 1e6:.1f} km²")

In [ ]:
# ---- Export to GeoPackage ----
valley_gdf = gpd.GeoDataFrame(
    {"method": ["VBET-simplified"],
     "gauge_id": [GAUGE_ID],
     "dem_resolution_m": [DEM_RESOLUTION],
     "hand_threshold_m_large": [VBET_CLASSES[0]["hand_m"]],
     "hand_threshold_m_medium": [VBET_CLASSES[1]["hand_m"]],
     "hand_threshold_m_small": [VBET_CLASSES[2]["hand_m"]],
     "area_km2": [valley_smooth.area / 1e6]},
    geometry=[valley_smooth],
    crs=CRS_PROJ,
)

valley_gdf.to_file(OUTPUT_GPKG, layer="valley_bottom", driver="GPKG")
print(f"Valley bottom saved to: {OUTPUT_GPKG}")

# Also save the HAND-threshold raster for inspection
valley_raster_path = DATA_DIR / f"cheyenne_valley_mask_{DEM_RESOLUTION}m.tif"
with rasterio.open(dem_path) as src:
    profile = src.profile.copy()
    profile.update(dtype="uint8", count=1, compress="deflate", nodata=0)
    with rasterio.open(valley_raster_path, "w", **profile) as dst:
        dst.write(valley_filled[np.newaxis, :, :])
print(f"Valley mask raster saved to: {valley_raster_path}")

---
## 7. Visualization & Validation

### 7a. Interactive Folium Map
Overlay the valley bottom polygon on a satellite basemap with NHD flowlines.

In [ ]:
# Reproject to WGS84 for Folium
valley_wgs84 = valley_gdf.to_crs(4326)
flowlines_wgs84 = flowlines_proj.to_crs(4326)

# Map center
bounds = valley_wgs84.total_bounds  # (minx, miny, maxx, maxy)
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

m = folium.Map(location=center, zoom_start=10, tiles=None)

# Satellite basemap
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite",
    overlay=False,
).add_to(m)

# Valley bottom polygon
folium.GeoJson(
    valley_wgs84,
    name="Valley Bottom (VBET)",
    style_function=lambda x: {
        "color": "#2ecc71", "weight": 1.5,
        "fillColor": "#2ecc71", "fillOpacity": 0.35,
    },
    tooltip=folium.GeoJsonTooltip(fields=["area_km2"], aliases=["Area (km²)"]),
).add_to(m)

# NHD Flowlines
folium.GeoJson(
    flowlines_wgs84,
    name="NHD Flowlines",
    style_function=lambda x: {"color": "#3498db", "weight": 1.5},
).add_to(m)

folium.LayerControl().add_to(m)
m

### 7b. Cross-Section Profiles

Plot elevation and HAND values along perpendicular transects to visually confirm that the
valley bottom threshold captures the floodplain and stops at the valley walls.

In [ ]:
import numpy as np
from shapely.geometry import LineString
from rasterio.sample import sample_gen

def sample_raster_along_line(raster_path, line, n_points=200):
    """Sample a raster along a Shapely LineString, return (distances, values)."""
    coords = [line.interpolate(i / n_points, normalized=True) for i in range(n_points + 1)]
    xy = [(c.x, c.y) for c in coords]
    with rasterio.open(raster_path) as src:
        values = np.array([v[0] for v in src.sample(xy)], dtype=np.float32)
        values = np.where(values == src.nodata, np.nan, values)
    dists = np.array([line.project(c) for c in coords]) / 1000  # → km
    return dists, values

# Build 3 transects perpendicular to the main stem flowline
# Use the main stem geometry (upstream flowline, largest segment by length)
main_stem = flowlines_proj.sort_values("geometry", key=lambda g: g.length, ascending=False).iloc[0]
ms_line = main_stem.geometry

# Sample transect positions at 25%, 50%, 75% along the main stem
transect_width_m = BUFFER_KM * 1000 * 1.5
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax_i, pct in enumerate([0.25, 0.5, 0.75]):
    pt = ms_line.interpolate(pct, normalized=True)

    # Perpendicular direction
    delta = 0.01
    pt_ahead = ms_line.interpolate(pct + delta, normalized=True)
    dx = pt_ahead.x - pt.x
    dy = pt_ahead.y - pt.y
    length = np.hypot(dx, dy)
    # Perpendicular: rotate 90°
    perp_dx = -dy / length * transect_width_m / 2
    perp_dy = dx / length * transect_width_m / 2

    transect = LineString([
        (pt.x - perp_dx, pt.y - perp_dy),
        (pt.x + perp_dx, pt.y + perp_dy),
    ])

    dists, elev = sample_raster_along_line(str(dem_path), transect)
    dists_h, hand_vals = sample_raster_along_line(str(hand_path), transect)

    ax = axes[ax_i]
    color_elev = "#795548"
    color_hand = "#1565C0"

    ax2 = ax.twinx()
    ax.plot(dists, elev, color=color_elev, lw=1.5, label="Elevation (m)")
    ax2.plot(dists_h, hand_vals, color=color_hand, lw=1.5, ls="--", label="HAND (m)")
    ax2.axhline(VBET_CLASSES[0]["hand_m"], color=color_hand, lw=0.8, ls=":",
                label=f"HAND threshold (large, {VBET_CLASSES[0]['hand_m']} m)")

    ax.set_title(f"Cross-section at {pct*100:.0f}% along main stem")
    ax.set_xlabel("Distance along transect (km)")
    ax.set_ylabel("Elevation (m)", color=color_elev)
    ax2.set_ylabel("HAND (m)", color=color_hand)
    ax.tick_params(axis="y", labelcolor=color_elev)
    ax2.tick_params(axis="y", labelcolor=color_hand)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

### 7c. Valley Bottom Area Statistics

In [ ]:
# Summary statistics
total_area_km2 = valley_gdf.geometry.area.sum() / 1e6
total_flowline_km = flowlines_proj.geometry.length.sum() / 1000
avg_width_m = (valley_gdf.geometry.area.sum() / flowlines_proj.geometry.length.sum())

print("=" * 50)
print("VALLEY BOTTOM SUMMARY")
print("=" * 50)
print(f"  Total area:          {total_area_km2:.1f} km²")
print(f"  Total flowline km:   {total_flowline_km:.1f} km")
print(f"  Average valley width:{avg_width_m:.0f} m")
print(f"  DEM resolution:      {DEM_RESOLUTION} m")
print(f"  Output file:         {OUTPUT_GPKG}")
print("=" * 50)
print()
print("Valley bottom polygon attributes:")
valley_gdf.drop(columns="geometry").T

---
## Notes & Next Steps

### Threshold Tuning
The HAND and slope thresholds in the configuration cell are starting points based on
literature values for semi-arid rivers. For the Cheyenne River specifically:
- Inspect cross-sections (Section 7b) — the valley bottom should capture the active
  floodplain and low terraces but stop at the first distinct break in slope
- If the polygon is too narrow, increase `hand_m` for the 'large' class
- If it spills onto uplands, decrease `hand_m` or `slope_deg`
- Compare visually against the satellite basemap in Section 7a

### CyVerse / Large-Area Runs
- For a full 300 km run, set `DEM_RESOLUTION = 30` for first pass, then re-run at 10 m
  for key sub-reaches
- Intermediate rasters (`cheyenne_dem_10m.tif`, `cheyenne_hand_10m.tif`) are cached to
  `data/` — re-running the notebook skips re-downloading/recomputing them
- On CyVerse, move these cached files to the Data Store to persist across sessions

### Downstream Use
The `data/cheyenne_valley_bottom.gpkg` polygon feeds directly into:
- `01_HLS_reflectance.ipynb` — clip HLS imagery to riparian corridor
- Future classification notebook — cottonwood / salt cedar / bare ground within valley bottom